In [41]:
# Clear all variables
%reset -f
%cd /home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors/

/home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors


In [42]:
from util import (
    JLMS_panel_technical_inefficiency_scores,
    NW_panel_technical_inefficiency_scores,
    direct_mapping_mat,
    inverse_mapping_vec,
    Loglikelihood_Gaussian_copula_cross_sectional_application_SFA,
    export_simulation_data_RS2007_electricity_application,
    estimate_Jondrow1982_u_hat,
    simulate_error_components,
    Estimate_NW_u_hat_conditional_W_cross_sectional_application,
 )
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
from scipy import stats
from scipy.linalg import expm, logm, norm
from scipy.optimize import minimize

In [43]:
#load parameters


theta = np.array([
    5.7630,
    0.1422,
    0.1170,
    0.0720,
    0.1851,
    0.5150,
    -0.0191,
    0.1561,
    0.0996,
    -0.0355,
    -0.1337,
    -0.1617,
    -0.1609,
    -0.0293,
    0.6475,
    2.0451
])

print(theta)

gamma = np.array([
4.691359544478445054e-01,
2.449463444476085439e-01,
4.738861695923803768e-02,
1.416673024522306368e-01,
2.191707371168801211e-01,
9.297518610871904343e-01,
7.927239502606412413e-01,
1.393802658718614085e+00,
3.252880911850812573e-02,
-4.430338547558972384e-01,
-9.527848585280578320e-01,
1.232980848191408789e-02,
-1.114131598725027317e+00,
2.370215935915223338e-01,
7.045769769525236370e-01
])
gamma

[ 5.763   0.1422  0.117   0.072   0.1851  0.515  -0.0191  0.1561  0.0996
 -0.0355 -0.1337 -0.1617 -0.1609 -0.0293  0.6475  2.0451]


array([ 0.46913595,  0.24494634,  0.04738862,  0.1416673 ,  0.21917074,
        0.92975186,  0.79272395,  1.39380266,  0.03252881, -0.44303385,
       -0.95278486,  0.01232981, -1.1141316 ,  0.23702159,  0.70457698])

In [44]:
ricefarm = pd.read_csv(r"ricefarm.csv")

ricefarm = ricefarm.iloc[:, [0, 1, 3, 5, 6, 7, 8, 14, 16]]

# Create ID dummies
dar = np.round(ricefarm["ID"] / 100000).astype(int)
id_dummies = pd.get_dummies(dar, prefix="DR").astype(int)
ricefarm = pd.concat([ricefarm, id_dummies.iloc[:, :-1]], axis=1)

# Create rice variety dummies
rice_dummies = pd.get_dummies(ricefarm.iloc[:, 2], prefix="VAR").astype(int)
ricefarm = pd.concat([ricefarm, rice_dummies.iloc[:, 1:]], axis=1)

# Recode TSP as logs and zeros
ricefarm[ricefarm.columns[5]] = np.where(
    ricefarm.iloc[:, 5] > 0, np.log(ricefarm.iloc[:, 5]), 0
)

# Convert pesticide usage to a dummy
ricefarm[ricefarm.columns[6]] = (ricefarm.iloc[:, 6] != 0).astype(int)

# Reorder the data
ricefarm = ricefarm.iloc[:, [8, 3, 4, 5, 7, 1, 6, 14, 15, 9, 10, 11, 12, 13]]

y = np.log(ricefarm.iloc[:, 0].values)
X = ricefarm.iloc[:, 1:].values

# Log covariates
X[:, 0:2] = np.log(X[:, 0:2])
X[:, 3] = np.log(X[:, 3])
X[:, 4] = np.log(X[:, 4])

# Cross sections of  y and X
N = 171
T = 6
y_t = []
X_t = []
for t in range(T):
    if t == 0:
        y_t.append(y[:N])
        X_t.append(X[:N, :])
    else:
        y_t.append(y[t * N : (t + 1) * N])
        X_t.append(X[t * N : (t + 1) * N, :])

# Import the model parameters from exercise 5.8
# theta = np.loadtxt("exercise_4_8_theta_python.txt", delimiter=",")

# Estimation of technical inefficiencies
# Get idx of percentile firms
percetile_firm_idx = np.round(np.arange(0.05, 1, 0.1) * N).astype(int)

# JLMS formula technical inefficiencies
JLMS_u_hat = JLMS_panel_technical_inefficiency_scores(theta=theta, y=y_t, X=X_t)
# Sort technical inefficiency scores by t=1 values
sort_idx = JLMS_u_hat[:, 0].argsort()
sorted_t1_JLMS_u_hat = JLMS_u_hat[sort_idx]
# Mean technical inefficiency for each firms over all t
JLMS_mean = np.mean(JLMS_u_hat, axis=1)
sorted_t1_JLMS_mean = JLMS_mean[sort_idx]
# Standard deviation of technical inefficiency for each firm over all t
JLMS_std = np.std(JLMS_u_hat, axis=1)
sorted_t1_JLMS_std = JLMS_std[sort_idx]

JLMS_u_hat_for_table = sorted_t1_JLMS_u_hat[percetile_firm_idx-1, :]
JLMS_std_u_hat_for_table = sorted_t1_JLMS_std[percetile_firm_idx-1]
JLMS_mean_u_hat_for_table = sorted_t1_JLMS_mean[percetile_firm_idx-1]

# Tabulate the technical inefficiency scores
JLMS_technical_inefficiency_results = pd.DataFrame(
    columns=[
        "Rank",
        "Fractile",
        "Std(u)",
        "Mean(u)",
        "t=1",
        "t=2",
        "t=3",
        "t=4",
        "t=5",
        "t=6",
    ]
)
JLMS_technical_inefficiency_results["Rank"] = percetile_firm_idx
JLMS_technical_inefficiency_results["Fractile"] = np.arange(0.05, 1, 0.1)
JLMS_technical_inefficiency_results["Std(u)"] = JLMS_std_u_hat_for_table
JLMS_technical_inefficiency_results["Mean(u)"] = JLMS_mean_u_hat_for_table
JLMS_technical_inefficiency_results.iloc[:, -6:] = JLMS_u_hat_for_table

# Compute averages
average_JLMS = pd.DataFrame(
    np.concatenate(
        [
            np.mean(JLMS_std_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_mean_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_u_hat_for_table, axis=0).reshape(-1, 1).T,
        ],
        axis=1,
    ),
    columns=["Std(u)", "Mean(u)", "t=1", "t=2", "t=3", "t=4", "t=5", "t=6"],
    index=["Average"],
)

print("JLMS Scores")
display(JLMS_technical_inefficiency_results)
display(average_JLMS)

JLMS Scores


/home/zwei1/.conda/envs/d2l_f25/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Rank,Fractile,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
0,9,0.05,0.199552,0.353685,0.155262,0.334548,0.751723,0.298472,0.168863,0.413244
1,26,0.15,0.266314,0.389385,0.19444,0.247792,0.901213,0.577862,0.251281,0.163724
2,43,0.25,0.089338,0.285361,0.22775,0.192722,0.460339,0.332544,0.232714,0.266099
3,60,0.35,0.139117,0.389762,0.258728,0.210491,0.295984,0.535175,0.477761,0.560436
4,77,0.45,0.200050,0.338565,0.283924,0.201444,0.559467,0.664157,0.178611,0.143785
5,94,0.55,0.065625,0.362564,0.307167,0.346151,0.296446,0.381004,0.349333,0.495284
6,111,0.65,0.108977,0.455180,0.336395,0.376161,0.39316,0.559239,0.642827,0.423299
7,128,0.75,0.139393,0.307534,0.393146,0.551261,0.307115,0.291799,0.134207,0.167675
8,145,0.85,0.293302,0.600276,0.466422,0.382638,1.180244,0.717366,0.284154,0.570831
9,162,0.95,0.138611,0.552418,0.568983,0.719867,0.540943,0.716203,0.425787,0.342725


,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
Average,0.164028,0.403473,0.319222,0.356307,0.568664,0.507382,0.314554,0.35471


In [45]:
ricefarm = pd.read_csv(r"ricefarm.csv")

ricefarm = ricefarm.iloc[:, [0, 1, 3, 5, 6, 7, 8, 14, 16]]

# Create ID dummies
dar = np.round(ricefarm["ID"] / 100000).astype(int)
id_dummies = pd.get_dummies(dar, prefix="DR").astype(int)
ricefarm = pd.concat([ricefarm, id_dummies.iloc[:, :-1]], axis=1)

# Create rice variety dummies
rice_dummies = pd.get_dummies(ricefarm.iloc[:, 2], prefix="VAR").astype(int)
ricefarm = pd.concat([ricefarm, rice_dummies.iloc[:, 1:]], axis=1)

# Recode TSP as logs and zeros
ricefarm[ricefarm.columns[5]] = np.where(
    ricefarm.iloc[:, 5] > 0, np.log(ricefarm.iloc[:, 5]), 0
)

# Convert pesticide usage to a dummy
ricefarm[ricefarm.columns[6]] = (ricefarm.iloc[:, 6] != 0).astype(int)

# Reorder the data
ricefarm = ricefarm.iloc[:, [8, 3, 4, 5, 7, 1, 6, 14, 15, 9, 10, 11, 12, 13]]

y = np.log(ricefarm.iloc[:, 0].values)
X = ricefarm.iloc[:, 1:].values

# Log covariates
X[:, 0:2] = np.log(X[:, 0:2])
X[:, 3] = np.log(X[:, 3])
X[:, 4] = np.log(X[:, 4])

# Cross sections of  y and X
N = 171
T = 6
y_t = []
X_t = []
for t in range(T):
    if t == 0:
        y_t.append(y[:N])
        X_t.append(X[:N, :])
    else:
        y_t.append(y[t * N : (t + 1) * N])
        X_t.append(X[t * N : (t + 1) * N, :])
        
# Import the model parameters from exercise 5.8
# theta = np.loadtxt("exercise_4_8_theta_python.txt", delimiter=",")
# gamma = np.loadtxt("exercise_4_8_gamma_python.txt", delimiter=",")

# Estimation of technical inefficiencies
# Get idx of percentile firms
percetile_firm_idx = np.round(np.arange(0.05, 1, 0.1) * N).astype(int)

# NW technical inefficiency estimates
Rho_hat = inverse_mapping_vec(gamma)
# Draw T dependent unfirm RV's for each s_{1}, ... 10000
Z = stats.multivariate_normal.rvs(
    size=10000, mean=np.zeros(T), cov=np.eye(T), random_state=1234
)
A = np.linalg.cholesky(Rho_hat)
U_hat = stats.norm.cdf(
    Z @ A, loc=np.zeros(T), scale=np.ones(T)
)  # Dependent uniform RVs from a gaussian copula
NW_u_hat = NW_panel_technical_inefficiency_scores(
    theta=theta, y=y_t, X=X_t, U_hat=U_hat
)

sort_idx = NW_u_hat[:, 0].argsort()
# Sort technical inefficiency scores by t=1 values
sorted_t1_NW_u_hat = NW_u_hat[sort_idx]
# Mean technical inefficiency for each firms over all t
NW_mean = np.mean(NW_u_hat, axis=1)
sorted_t1_NW_mean = NW_mean[sort_idx]
# Standard deviation of technical inefficiency for each firm over all t
NW_std = np.std(NW_u_hat, axis=1)
sorted_t1_NW_std = NW_std[sort_idx]

NW_u_hat_for_table = sorted_t1_NW_u_hat[percetile_firm_idx-1, :]
NW_std_u_hat_for_table = sorted_t1_NW_std[percetile_firm_idx-1]
NW_mean_u_hat_for_table = sorted_t1_NW_mean[percetile_firm_idx-1]

NW_technical_inefficiency_results = pd.DataFrame(
    columns=[
        "Rank",
        "Fractile",
        "Std(u)",
        "Mean(u)",
        "t=1",
        "t=2",
        "t=3",
        "t=4",
        "t=5",
        "t=6",
    ]
)
NW_technical_inefficiency_results["Rank"] = percetile_firm_idx
NW_technical_inefficiency_results["Fractile"] = np.arange(0.05, 1, 0.1)
NW_technical_inefficiency_results["Std(u)"] = NW_std_u_hat_for_table
NW_technical_inefficiency_results["Mean(u)"] = NW_mean_u_hat_for_table
NW_technical_inefficiency_results.iloc[:, -6:] = NW_u_hat_for_table

# Compute averages
average_NW = pd.DataFrame(
    np.concatenate(
        [
            np.mean(NW_std_u_hat_for_table).reshape(-1, 1).T,
            np.mean(NW_mean_u_hat_for_table).reshape(-1, 1).T,
            np.mean(NW_u_hat_for_table, axis=0).reshape(-1, 1).T,
        ],
        axis=1,
    ),
    columns=["Std(u)", "Mean(u)", "t=1", "t=2", "t=3", "t=4", "t=5", "t=6"],
    index=["Average"],
)

print("Nadaraya Watson Scores")
display(NW_technical_inefficiency_results)
display(average_NW)

/home/zwei1/.conda/envs/d2l_f25/lib/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Nadaraya Watson Scores


,Rank,Fractile,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
0,9,0.05,0.245780,0.403574,0.024189,0.318616,0.738029,0.688577,0.270438,0.381592
1,26,0.15,0.169354,0.239335,0.052607,0.124727,0.510406,0.431043,0.138368,0.17886
2,43,0.25,0.136559,0.232430,0.09543,0.093579,0.367388,0.457905,0.165529,0.214747
3,60,0.35,0.167328,0.237828,0.158177,0.10352,0.195613,0.605008,0.161773,0.202875
4,77,0.45,0.185032,0.224482,0.195306,0.182155,0.163092,0.627522,0.095727,0.083093
5,94,0.55,0.145018,0.196599,0.227689,0.126032,0.506266,0.104792,0.108692,0.106124
6,111,0.65,0.100930,0.287980,0.290795,0.273055,0.250433,0.501722,0.218845,0.193028
7,128,0.75,0.236154,0.332206,0.34082,0.140553,0.474093,0.773594,0.094968,0.169209
8,145,0.85,0.273121,0.407474,0.465053,0.11628,0.9142,0.542475,0.17906,0.227774
9,162,0.95,0.223640,0.543871,0.667484,0.659475,0.313284,0.918952,0.307447,0.396586


,Std(u),Mean(u),t=1,t=2,t=3,t=4,t=5,t=6
Average,0.188292,0.310578,0.251755,0.213799,0.44328,0.565159,0.174085,0.215389


In [46]:
data = pd.read_excel('steamelectric.xlsx')

data['fuel_price_index'] = np.nan
data.iloc[:-1, -1] = data.iloc[:-1, 2].values/data.iloc[1:,2].values
data['LM_price_index'] = np.nan
data.iloc[:-1, -1] = data.iloc[:-1, 3].values/data.iloc[1:,3].values

data.iloc[13:1007:14, :] = np.nan #drop last fuel price index for each plant
data = data.dropna()


In [ ]:

X1 = (data['Fuel Costs ($1000)']/data['fuel_price_index'])*1000*1e-6; #fuel costs over price index
X2 = data['Operating Costs ($1000)']/data['LM_price_index']*1000*1e-6; #LM costs over price index
X3 = data['Capital ($1000)']/1000; # capital

X = np.log(pd.concat([X1, X2, X3], axis=1))
P = np.log(data[['fuel_price_index', 'LM_price_index', 'User Cost of Capital']])
y = np.log(data['Output (MWhr)']*1e-6) # output ml MWpH 

# Estimate cross sectional models
n = len(y)
S = 500 #500 Number of Halton draws used in maximum simulated likelihood
S_kernel = 10000; #10000 Number of simulated draws for evaluation of the conditional expectation
n_inputs = 3
n_corr_terms = ((n_inputs)^2-(n_inputs))/2 # Number of off diagonal lower triangular correlation/covariance terms for Gaussian copula correlation matrix. 

initial_lalpha = -2.6
initial_beta = np.array([0.5, 0.3, 0.3])
initial_lbeta = np.log(initial_beta)
initial_logit_beta = np.log(initial_beta/(1-initial_beta))
initial_sigma2_v = 0.015
initial_lsigma2_v = np.log(initial_sigma2_v)
initial_sigma2_u = 0.15
initial_lsigma2_u = np.log(initial_sigma2_u)
initial_mu_W = np.array([0.5, 0])

sampler = stats.qmc.Halton(d=1, 
                           scramble=True, 
                           seed=123)
sample = sampler.random(n=n)
us_ = stats.norm.ppf((sample+1)/2, 0, 1)
us_Sxn = np.reshape(np.repeat(us_[:, np.newaxis], S, axis=1), (S, n))

# Gaussian Copula
initial_Sigma = np.array([[0.2, 0.09], 
                          [0.09, 0.2]])
initial_sigma2_w = np.diag(initial_Sigma)
initial_lsigma2_w = np.log(initial_sigma2_w)

eps_ = y - initial_lalpha - X@initial_beta
W_ = (np.tile(X.iloc[:, 0].values.reshape(-1, 1), (1, n_inputs-1)) - X.iloc[:,1:].values) - (P.iloc[:,1:].values - np.tile(P.iloc[:, 0].values.reshape(-1, 1), (1, n_inputs-1)) +(np.log(initial_beta[0]) - np.log(initial_beta[1:])))
initial_Rho = np.corrcoef(np.concatenate([eps_.values.reshape(-1,1), W_], axis=1).T)
initial_lRho = direct_mapping_mat(initial_Rho)

Gaussian_copula_theta0 = np.concatenate([np.array([initial_lalpha]), initial_lbeta, np.array([initial_sigma2_v, initial_sigma2_u]), 
                                         initial_lsigma2_w, initial_mu_W, initial_lRho])

# Bounds to ensure sigma2v and sigma2u are >= 0
bounds = [(None, None) for x in range(4)] + [
    (1e-5, np.inf),
    (1e-5, np.inf),
] + [(None, None) for x in range(7)]

# Minimize the negative log-likelihood using numerical optimization.
MLE_results = minimize(
    fun=Loglikelihood_Gaussian_copula_cross_sectional_application_SFA,
    x0=Gaussian_copula_theta0,
    method="L-BFGS-B",
    tol = 1e-6,
    options={"ftol": 1e-6, "maxiter": 1000, "maxfun": 6*1000},
    args=(y.values, X.values, P.values, us_Sxn, n_inputs, S),
    bounds=bounds,
)

Gaussian_copula_theta = MLE_results.x
logMLE = MLE_results.fun * -1  # Log-likelihood at the solution

# Transform parameters
Gaussian_copula_theta[:4] = np.exp(Gaussian_copula_theta[:4]) # Transform production system coefficients
Gaussian_copula_theta[6:8] = np.exp(Gaussian_copula_theta[6:8])
rhos_log_form = Gaussian_copula_theta[-3:]
Rho = inverse_mapping_vec(rhos_log_form)
Gaussian_copula_theta[-3:] = direct_mapping_mat(initial_Rho)
Rho_lower_trianglular = Rho[np.tril_indices(Rho.shape[1], 1)]

np.savetxt(r'./cross_sectional_SFA_RS2007_electricity_application_gaussian_copula_correlation_matrix.csv', 
              Rho, 
              delimiter=',')

# JLMS technical inefficiency scores
(Gaussian_copula_JLMS_u_hat, 
 Gaussian_copula_JLMS_V_u_hat) = estimate_Jondrow1982_u_hat(theta=Gaussian_copula_theta, 
                                                            n_inputs=n_inputs, 
                           n_corr_terms=n_corr_terms, 
                           y=y.values, 
                           X=X.values)
JLMS_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_JLMS_u_hat.reshape(-1,1)], axis=1)
JLMS_V_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_JLMS_V_u_hat.reshape(-1,1)], axis=1)
JLMS_u_hat_year_mean = np.zeros((13, 2))
JLMS_V_u_hat_year_mean = np.zeros((13, 2))
JLMS_std_u_hat_year_mean = np.zeros((13, 2))
for i, t in enumerate(range(86, 99)):
    JLMS_u_hat_year_mean[i, 0] = t
    JLMS_V_u_hat_year_mean[i, 0] = t
    JLMS_std_u_hat_year_mean[i, 0] = t
    JLMS_u_hat_year_mean[i, 1] = np.mean(JLMS_u_hat_matrix[JLMS_u_hat_matrix[:,0] == t, 1])
    JLMS_V_u_hat_year_mean[i, 1] = np.mean(JLMS_V_u_hat_matrix[JLMS_V_u_hat_matrix[:,0] == t, 1])
    JLMS_std_u_hat_year_mean[i, 1:] = np.mean(np.sqrt(JLMS_V_u_hat_matrix[JLMS_V_u_hat_matrix[:,0] == t, 1]))

# Copula Nadaraya Watson Scores
Gaussian_copula_U_hat = simulate_error_components(Rho=Rho,
                                                                      n_inputs=n_inputs, 
                                                                      S_kernel=S_kernel, 
                                                                      seed=1234)
(Gaussian_copula_NW_conditional_W_u_hat, 
 Gaussian_copula_NW_conditional_W_V_u_hat) = Estimate_NW_u_hat_conditional_W_cross_sectional_application(theta=Gaussian_copula_theta, 
                                                                                                         n_inputs=n_inputs, 
                                                                                                         n_corr_terms=n_corr_terms, 
                                                                                                         y=y.values, 
                                                                                                         X=X.values, 
                                                                                                         P=P.values, 
                                                                                                 U_hat=Gaussian_copula_U_hat, 
                                                                                                         S_kernel=S_kernel)
NW_conditional_W_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_NW_conditional_W_u_hat.reshape(-1,1)], axis=1)
NW_conditional_W_V_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_NW_conditional_W_V_u_hat.reshape(-1,1)], axis=1)
NW_conditional_W_u_hat_year_mean = np.zeros((13, 2))
NW_conditional_W_V_u_hat_year_mean = np.zeros((13, 2))
NW_conditional_W_std_u_hat_year_mean = np.zeros((13, 2))
for i, t in enumerate(range(86, 99)):
    NW_conditional_W_u_hat_year_mean[i, 0] = t
    NW_conditional_W_V_u_hat_year_mean[i, 0] = t
    NW_conditional_W_std_u_hat_year_mean[i, 0] = t
    NW_conditional_W_u_hat_year_mean[i, 1] = np.mean(NW_conditional_W_u_hat_matrix[NW_conditional_W_u_hat_matrix[:,0] == t, 1])
    NW_conditional_W_V_u_hat_year_mean[i, 1] = np.mean(NW_conditional_W_V_u_hat_matrix[NW_conditional_W_V_u_hat_matrix[:,0] == t, 1])
    NW_conditional_W_std_u_hat_year_mean[i, 1] = np.mean(np.sqrt(NW_conditional_W_V_u_hat_matrix[NW_conditional_W_V_u_hat_matrix[:,0] == t, 1]))

# Export simulated training data and compute LLF u hat
export_simulation_data_RS2007_electricity_application(theta=Gaussian_copula_theta, 
                                                      n_inputs=n_inputs, 
                                                      y=y.values, 
                                                      X=X.values, 
                                                      P=P.values, 
                                                      U_hat=Gaussian_copula_U_hat, 
                                                      S_kernel=S_kernel)

# Users should change the first directory to the path where R is installed - use the code depending on your OS
# Windows
# subprocess.call([r'C:\Program Files\R\R-4.2.2\bin\Rscript.exe ./train_LocalLinear_forest_cross_sectional_RS2007_electricty_application.R'], 
#                 shell=True)
# Mac
# subprocess.call([r'/usr/local/bin/Rscript', './train_LocalLinear_forest_cross_sectional_RS2007_electricty_application.R'])
Gaussian_copula_LLF_results = pd.read_csv(r'./LLF_Gaussian_copula_u_hat.csv')
Gaussian_copula_LLF_conditional_W_u_hat = Gaussian_copula_LLF_results.iloc[:,0].values
Gaussian_copula_LLF_conditional_W_V_u_hat = Gaussian_copula_LLF_results.iloc[:,1].values

LLF_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_LLF_conditional_W_u_hat.reshape(-1,1)], axis=1)
LLF_V_u_hat_matrix = np.concatenate([data.loc[:, ['YEAR']].values, Gaussian_copula_LLF_conditional_W_V_u_hat.reshape(-1,1)], axis=1)
LLF_u_hat_year_mean = np.zeros((13, 2))
LLF_V_u_hat_year_mean = np.zeros((13, 2))
LLF_std_u_hat_year_mean = np.zeros((13, 2))
for i, t in enumerate(range(86, 99)):
    LLF_u_hat_year_mean[i, 0] = t
    LLF_V_u_hat_year_mean[i, 0] = t
    LLF_std_u_hat_year_mean[i, 0] = t
    LLF_u_hat_year_mean[i, 1] = np.mean(LLF_u_hat_matrix[LLF_u_hat_matrix[:,0] == t, 1])
    LLF_V_u_hat_year_mean[i, 1] = np.mean(LLF_V_u_hat_matrix[LLF_V_u_hat_matrix[:,0] == t, 1])
    LLF_std_u_hat_year_mean[i, 1] = np.mean(np.sqrt(LLF_V_u_hat_matrix[LLF_V_u_hat_matrix[:,0] == t, 1]))

mean_Gaussian_copula_JLMS_u_hat = np.mean(Gaussian_copula_JLMS_u_hat)
mean_Gaussian_copula_JLMS_std_u_hat = np.mean(JLMS_std_u_hat_year_mean[:,1])
mean_Gaussian_copula_NW_conditional_W_u_hat = np.mean(Gaussian_copula_NW_conditional_W_u_hat)
mean_Gaussian_copula_NW_conditional_W_std_u_hat = np.mean(NW_conditional_W_std_u_hat_year_mean[:,1])
mean_Gaussian_copula_LLF_conditional_W_u_hat = np.mean(Gaussian_copula_LLF_conditional_W_u_hat);
mean_Gaussian_copula_LLF_conditional_W_std_u_hat = np.mean(LLF_std_u_hat_year_mean[:,1])

cross_sectional_results_table = pd.DataFrame(np.concatenate([np.array([x for x in range(86, 99)]).reshape(-1,1), 
                                                JLMS_u_hat_year_mean[:, [1]], JLMS_std_u_hat_year_mean[:, [1]], 
                                                NW_conditional_W_u_hat_year_mean[:, [1]], NW_conditional_W_std_u_hat_year_mean[:, [1]], 
                                                LLF_u_hat_year_mean[:, [1]], LLF_std_u_hat_year_mean[:, [1]]], axis=1), 
                                             columns=['Year', 'JLMS Est.', 'JLMS Std Dev', 'NW Est.', 'NW Std Dev', 'LLF Est.', 'LLF Std Dev'])
cross_sectional_averages = pd.DataFrame(np.array([mean_Gaussian_copula_JLMS_u_hat, 
                                     mean_Gaussian_copula_JLMS_std_u_hat, 
                                     mean_Gaussian_copula_NW_conditional_W_u_hat, 
                                     mean_Gaussian_copula_NW_conditional_W_std_u_hat, 
                                     mean_Gaussian_copula_LLF_conditional_W_u_hat, 
                                     mean_Gaussian_copula_LLF_conditional_W_std_u_hat]).reshape(1,-1), 
                                        columns=['JLMS Est.', 'JLMS Std Dev', 'NW Est.', 'NW Std Dev', 'LLF Est.', 'LLF Std Dev'], 
                                        index=['Average'])
print('Cross-Sectional Models (APS16 Estimator)')
display(cross_sectional_results_table)
display(cross_sectional_averages)

In [48]:
data

,Firm No,YEAR,Fuel Price ($/BTU),Price of Labor and Maintenance,User Cost of Capital,Fuel Quantities (1000 BTU),Aggregate of Labor and Maintenance ($1000),Capital ($1000),Output (MWhr),Fuel Costs ($1000),Operating Costs ($1000),Variable cost ($1000),Variable cost ($10+7),Total cost ($10+7),Output (MWhr)*10+6,fuel_price_index,LM_price_index
0,1.0,86.0,2.112398,17.751674,0.12718,145496.221477,4080.291162,1.022291e+06,17472940.0,307346.0,72432.0,379778.0,37.9778,50.979295,17.472940,1.024904,1.005947
1,1.0,87.0,2.061070,17.646729,0.13177,147975.576035,4775.049313,1.007952e+06,18525740.0,304988.0,84264.0,389252.0,38.9252,52.206983,18.525740,1.085470,0.934921
2,1.0,88.0,1.898781,18.875110,0.13350,158366.323067,4602.198307,1.003405e+06,19542630.0,300703.0,86867.0,387570.0,38.7570,52.152456,19.542630,0.927553,0.991760
3,1.0,89.0,2.047088,19.031942,0.13081,150056.113351,4531.066794,1.004536e+06,20041930.0,307178.0,86235.0,393413.0,39.3413,52.481638,20.041930,0.929214,1.021844
4,1.0,90.0,2.203031,18.625089,0.13867,149636.120305,4934.956369,9.940630e+05,21361100.0,329653.0,91914.0,421567.0,42.1567,55.941372,21.361100,1.045851,0.932024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,72.0,94.0,2.033161,25.748563,0.11670,49578.954604,1643.509214,3.007979e+05,7047510.0,100802.0,42318.0,143120.0,14.3120,17.822312,7.047510,1.068363,0.964506
1003,72.0,95.0,1.903062,26.696107,0.11104,49458.179311,1720.063501,3.044827e+05,7428612.0,94122.0,45919.0,140041.0,14.0041,17.385076,7.428612,0.930110,0.939520
1004,72.0,96.0,2.046062,28.414624,0.10887,47930.120221,1451.330118,3.025390e+05,7956378.0,98068.0,41239.0,139307.0,13.9307,17.224442,7.956378,0.998571,1.001333
1005,72.0,97.0,2.048989,28.376800,0.10798,48360.424322,1131.029573,2.992974e+05,8213518.0,99090.0,32095.0,131186.0,13.1186,16.350413,8.213518,1.018678,0.976188


In [ ]:
bootstrap_reps = 20
bootstrap_seed = 123
bootstrap_cluster_column = "Firm No"


def _stabilize_correlation_matrix(matrix):
    matrix = np.asarray(matrix, dtype=float)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    matrix = (matrix + matrix.T) / 2
    eigvals = np.linalg.eigvalsh(matrix)
    min_eig = eigvals.min()
    if min_eig <= 1e-8:
        matrix = matrix + np.eye(matrix.shape[0]) * (1e-8 - min_eig + 1e-8)
    scales = np.sqrt(np.diag(matrix))
    matrix = matrix / np.outer(scales, scales)
    np.fill_diagonal(matrix, 1.0)
    return matrix


def _cluster_bootstrap_sample(df, cluster_col, random_state):
    if cluster_col not in df.columns:
        raise KeyError(f"'{cluster_col}' not found in data columns")

    rng = np.random.default_rng(random_state)
    unique_clusters = df[cluster_col].dropna().unique()
    sampled_clusters = rng.choice(unique_clusters, size=len(unique_clusters), replace=True)

    sampled_frames = []
    for draw_idx, cluster_value in enumerate(sampled_clusters):
        cluster_rows = df[df[cluster_col] == cluster_value].copy()
        cluster_rows["bootstrap_firm_draw"] = draw_idx
        sampled_frames.append(cluster_rows)

    return pd.concat(sampled_frames, ignore_index=True)


def _bootstrap_cross_sectional_means(sampled_data, seed):
    data_boot = sampled_data.copy()

    X1 = (data_boot["Fuel Costs ($1000)"] / data_boot["fuel_price_index"]) * 1000 * 1e-6
    X2 = data_boot["Operating Costs ($1000)"] / data_boot["LM_price_index"] * 1000 * 1e-6
    X3 = data_boot["Capital ($1000)"] / 1000

    X = np.log(pd.concat([X1, X2, X3], axis=1))
    P = np.log(data_boot[["fuel_price_index", "LM_price_index", "User Cost of Capital"]])
    y = np.log(data_boot["Output (MWhr)"] * 1e-6)

    n = len(y)
    S = 500
    S_kernel = 10000
    n_inputs = 3
    n_corr_terms = ((n_inputs) ** 2 - (n_inputs)) / 2

    initial_lalpha = -2.6
    initial_beta = np.array([0.5, 0.3, 0.3])
    initial_lbeta = np.log(initial_beta)
    initial_sigma2_v = 0.015
    initial_sigma2_u = 0.15
    initial_mu_W = np.array([0.5, 0])

    sampler = stats.qmc.Halton(d=1, scramble=True, seed=seed)
    sample = sampler.random(n=n)
    us_ = stats.norm.ppf((sample + 1) / 2, 0, 1)
    us_Sxn = np.reshape(np.repeat(us_[:, np.newaxis], S, axis=1), (S, n))

    initial_Sigma = np.array([[0.2, 0.09], [0.09, 0.2]])
    initial_sigma2_w = np.diag(initial_Sigma)
    initial_lsigma2_w = np.log(initial_sigma2_w)

    eps_ = y - initial_lalpha - X @ initial_beta
    W_ = (
        np.tile(X.iloc[:, 0].values.reshape(-1, 1), (1, n_inputs - 1)) - X.iloc[:, 1:].values
    ) - (
        P.iloc[:, 1:].values
        - np.tile(P.iloc[:, 0].values.reshape(-1, 1), (1, n_inputs - 1))
        + (np.log(initial_beta[0]) - np.log(initial_beta[1:]))
    )
    initial_Rho = np.corrcoef(np.concatenate([eps_.values.reshape(-1, 1), W_], axis=1).T)
    initial_Rho = _stabilize_correlation_matrix(initial_Rho)
    initial_lRho = direct_mapping_mat(initial_Rho)

    Gaussian_copula_theta0 = np.concatenate(
        [
            np.array([initial_lalpha]),
            initial_lbeta,
            np.array([initial_sigma2_v, initial_sigma2_u]),
            initial_lsigma2_w,
            initial_mu_W,
            initial_lRho,
        ]
    )

    bounds = [(None, None) for _ in range(4)] + [
        (1e-5, np.inf),
        (1e-5, np.inf),
    ] + [(None, None) for _ in range(7)]

    MLE_results = minimize(
        fun=Loglikelihood_Gaussian_copula_cross_sectional_application_SFA,
        x0=Gaussian_copula_theta0,
        method="L-BFGS-B",
        tol=1e-6,
        options={"ftol": 1e-6, "maxiter": 1000, "maxfun": 6 * 1000},
        args=(y.values, X.values, P.values, us_Sxn, n_inputs, S),
        bounds=bounds,
    )

    Gaussian_copula_theta = MLE_results.x
    logMLE = MLE_results.fun * -1

    Gaussian_copula_theta[:4] = np.exp(Gaussian_copula_theta[:4])
    Gaussian_copula_theta[6:8] = np.exp(Gaussian_copula_theta[6:8])
    rhos_log_form = Gaussian_copula_theta[-3:]
    Rho = inverse_mapping_vec(rhos_log_form)
    Gaussian_copula_theta[-3:] = direct_mapping_mat(initial_Rho)
    Rho_lower_trianglular = Rho[np.tril_indices(Rho.shape[1], 1)]

    Gaussian_copula_JLMS_u_hat, Gaussian_copula_JLMS_V_u_hat = estimate_Jondrow1982_u_hat(
        theta=Gaussian_copula_theta,
        n_inputs=n_inputs,
        n_corr_terms=n_corr_terms,
        y=y.values,
        X=X.values,
    )
    JLMS_u_hat_matrix = np.concatenate(
        [data_boot.loc[:, ["YEAR"]].values, Gaussian_copula_JLMS_u_hat.reshape(-1, 1)], axis=1
    )
    JLMS_V_u_hat_matrix = np.concatenate(
        [data_boot.loc[:, ["YEAR"]].values, Gaussian_copula_JLMS_V_u_hat.reshape(-1, 1)], axis=1
    )
    JLMS_u_hat_year_mean = np.zeros((13, 2))
    JLMS_V_u_hat_year_mean = np.zeros((13, 2))
    JLMS_std_u_hat_year_mean = np.zeros((13, 2))
    for i, t in enumerate(range(86, 99)):
        JLMS_u_hat_year_mean[i, 0] = t
        JLMS_V_u_hat_year_mean[i, 0] = t
        JLMS_std_u_hat_year_mean[i, 0] = t
        JLMS_u_hat_year_mean[i, 1] = np.mean(JLMS_u_hat_matrix[JLMS_u_hat_matrix[:, 0] == t, 1])
        JLMS_V_u_hat_year_mean[i, 1] = np.mean(JLMS_V_u_hat_matrix[JLMS_V_u_hat_matrix[:, 0] == t, 1])
        JLMS_std_u_hat_year_mean[i, 1] = np.mean(np.sqrt(JLMS_V_u_hat_matrix[JLMS_V_u_hat_matrix[:, 0] == t, 1]))

    Gaussian_copula_U_hat = simulate_error_components(
        Rho=Rho,
        n_inputs=n_inputs,
        S_kernel=S_kernel,
        seed=1234 + seed,
    )
    (
        Gaussian_copula_NW_conditional_W_u_hat,
        Gaussian_copula_NW_conditional_W_V_u_hat,
    ) = Estimate_NW_u_hat_conditional_W_cross_sectional_application(
        theta=Gaussian_copula_theta,
        n_inputs=n_inputs,
        n_corr_terms=n_corr_terms,
        y=y.values,
        X=X.values,
        P=P.values,
        U_hat=Gaussian_copula_U_hat,
        S_kernel=S_kernel,
    )
    NW_conditional_W_u_hat_matrix = np.concatenate(
        [data_boot.loc[:, ["YEAR"]].values, Gaussian_copula_NW_conditional_W_u_hat.reshape(-1, 1)], axis=1
    )
    NW_conditional_W_V_u_hat_matrix = np.concatenate(
        [data_boot.loc[:, ["YEAR"]].values, Gaussian_copula_NW_conditional_W_V_u_hat.reshape(-1, 1)], axis=1
    )
    NW_conditional_W_u_hat_year_mean = np.zeros((13, 2))
    NW_conditional_W_V_u_hat_year_mean = np.zeros((13, 2))
    NW_conditional_W_std_u_hat_year_mean = np.zeros((13, 2))
    for i, t in enumerate(range(86, 99)):
        NW_conditional_W_u_hat_year_mean[i, 0] = t
        NW_conditional_W_V_u_hat_year_mean[i, 0] = t
        NW_conditional_W_std_u_hat_year_mean[i, 0] = t
        NW_conditional_W_u_hat_year_mean[i, 1] = np.mean(
            NW_conditional_W_u_hat_matrix[NW_conditional_W_u_hat_matrix[:, 0] == t, 1]
        )
        NW_conditional_W_V_u_hat_year_mean[i, 1] = np.mean(
            NW_conditional_W_V_u_hat_matrix[NW_conditional_W_V_u_hat_matrix[:, 0] == t, 1]
        )
        NW_conditional_W_std_u_hat_year_mean[i, 1] = np.mean(
            np.sqrt(NW_conditional_W_V_u_hat_matrix[NW_conditional_W_V_u_hat_matrix[:, 0] == t, 1])
        )

    mean_Gaussian_copula_JLMS_u_hat = np.mean(Gaussian_copula_JLMS_u_hat)
    mean_Gaussian_copula_JLMS_std_u_hat = np.mean(JLMS_std_u_hat_year_mean[:, 1])
    mean_Gaussian_copula_NW_conditional_W_u_hat = np.mean(Gaussian_copula_NW_conditional_W_u_hat)
    mean_Gaussian_copula_NW_conditional_W_std_u_hat = np.mean(NW_conditional_W_std_u_hat_year_mean[:, 1])

    return {
        "mean_Gaussian_copula_JLMS_u_hat": mean_Gaussian_copula_JLMS_u_hat,
        "mean_Gaussian_copula_JLMS_std_u_hat": mean_Gaussian_copula_JLMS_std_u_hat,
        "mean_Gaussian_copula_NW_conditional_W_u_hat": mean_Gaussian_copula_NW_conditional_W_u_hat,
        "mean_Gaussian_copula_NW_conditional_W_std_u_hat": mean_Gaussian_copula_NW_conditional_W_std_u_hat,
        "mean_Gaussian_copula_LLF_conditional_W_u_hat": np.nan,
        "mean_Gaussian_copula_LLF_conditional_W_std_u_hat": np.nan,
        "logMLE": logMLE,
        "optimization_success": MLE_results.success,
        "optimization_message": MLE_results.message,
    }


bootstrap_records = []
for b in range(bootstrap_reps):
    sampled_data = _cluster_bootstrap_sample(
        data,
        cluster_col=bootstrap_cluster_column,
        random_state=bootstrap_seed + b,
    )
    try:
        bootstrap_records.append(
            _bootstrap_cross_sectional_means(sampled_data=sampled_data, seed=bootstrap_seed + b)
        )
    except Exception as exc:
        bootstrap_records.append(
            {
                "mean_Gaussian_copula_JLMS_u_hat": np.nan,
                "mean_Gaussian_copula_JLMS_std_u_hat": np.nan,
                "mean_Gaussian_copula_NW_conditional_W_u_hat": np.nan,
                "mean_Gaussian_copula_NW_conditional_W_std_u_hat": np.nan,
                "mean_Gaussian_copula_LLF_conditional_W_u_hat": np.nan,
                "mean_Gaussian_copula_LLF_conditional_W_std_u_hat": np.nan,
                "logMLE": np.nan,
                "optimization_success": False,
                "optimization_message": str(exc),
            }
        )

bootstrap_results = pd.DataFrame(bootstrap_records)
bootstrap_summary = bootstrap_results[[
    "mean_Gaussian_copula_JLMS_u_hat",
    "mean_Gaussian_copula_NW_conditional_W_u_hat",
    "mean_Gaussian_copula_LLF_conditional_W_u_hat",
]].describe().T

display(bootstrap_results)
display(bootstrap_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_columns = [
    "mean_Gaussian_copula_JLMS_u_hat",
    "mean_Gaussian_copula_NW_conditional_W_u_hat",
    "mean_Gaussian_copula_LLF_conditional_W_u_hat",
]
plot_titles = ["Bootstrap JLMS Means", "Bootstrap NW Means", "Bootstrap LLF Means"]

for ax, column, title in zip(axes, plot_columns, plot_titles):
    bootstrap_results[column].dropna().hist(ax=ax, bins=10)
    ax.set_title(title)
    ax.set_xlabel("Mean estimate")
    ax.set_ylabel("Frequency")

plt.tight_layout()
print("Bootstrap sampling is clustered on Firm No.")
print("The bootstrap cell applies your Python estimation block to each bootstrap sample.")
print("LLF is left as NaN because you asked not to use the R-based LLF workflow.")